# IDB — Chile climate projects → city actions (idb-projects, v1)

Standalone review notebook for the **IDB** source: pull Chile operations from the IDB open-data datastore, keep the **mitigation** ones (keyword-scored), standardise the fields, and map each to the city action list at the sector/subsector grain. Awards/projects layer, multilateral level, intermediated access. Writes two committed `data/` outputs: the Chile project dataset and the project→action crosswalk. The only external reference is the city action list; everything else is in this folder. Mediums/lows/none **await human adjudication**.

In [ ]:
%matplotlib inline
import os, json, requests, pandas as pd
ACTIONS='../../../../cl-ssg/cl-ssg-projects/releases/v1/data/derived/actions_profiled.csv'   # the city action list (the one allowed cross-review reference)
assert os.path.exists(ACTIONS), ACTIONS
acts=pd.read_csv(ACTIONS); acts['id']=acts[acts.columns[0]]
acat={r['id']:(str(r['action_name']).strip(), r['gpc_reference_number']) for _,r in acts.iterrows()}
len(acat)

In [ ]:
# Extract: IDB datastore (Chile filter via CKAN filters), paginated
url='https://data.iadb.org/api/action/datastore_search'; rid='814b7b54-477a-4c25-b3bf-6be05412069d'
recs=[]; off=0
while True:
    d=requests.get(url, params={'resource_id':rid,'limit':1000,'offset':off,'filters':json.dumps({'cntry_cd':'CL'})}, timeout=60).json()
    r=d['result']['records']
    if not r: break
    recs+=r; off+=1000
    if len(r)<1000: break
RENAME={'oper_num':'project_id','oper_nm':'project_name','objtv':'objective','cntry_cd':'country_code','cntry_nm':'country_name','sector_nm':'sector','subsector_nm':'subsector','apprvl_dt':'approval_date','publc_sts_nm':'status','orig_apprvd_useq_amnt':'total_commitment_amount','lending_instrmnt_nm':'instrument_type'}
raw=pd.DataFrame(recs).rename(columns=RENAME)
print('Chile IDB operations:', len(raw))
raw.head()

In [ ]:
# Keep mitigation operations (keyword score on name + objective + sector + subsector)
MIT=['renewable','energy efficiency','solar','wind','electromobility','electric bus','electric vehicle','recycl','waste','forest','reforest','redd','low-carbon','emission','clean energy','biogas','hydrogen']
def is_mit(r):
    t=' '.join(str(r.get(c,'')) for c in ['project_name','objective','sector','subsector']).lower()
    return any(k in t for k in MIT)
raw['is_mitigation']=raw.apply(is_mit, axis=1)
keep=['source','project_id','project_name','objective','country_name','sector','subsector','approval_date','status','total_commitment_amount','instrument_type','is_mitigation']
raw['source']='IDB'
chile=raw[raw['is_mitigation']][[c for c in keep if c in raw.columns]].copy()
print('mitigation operations:', len(chile), 'of', len(raw))
chile.head()

In [ ]:
# Map each project to a city action (sector/theme/name text, link-don't-attribute)
KW=[
  (['electric bus','e-bus','zero-emission bus','bus rapid','brt'], 'c40_0023','high','Electric/zero-emission buses -> adopt zero-emission bus fleets.'),
  (['e-mobility','electromobility','electric vehicle',' ev ','zero-emission transport'], 'c40_0023','medium','Electromobility / EV fleets -> zero-emission fleets (transport).'),
  (['solar','photovoltaic',' pv '], 'icare_0012','medium','Solar generation -> solar on public assets (utility-scale only a partial fit).'),
  (['energy efficiency','retrofit','thermal','insulation'], 'c40_0016','medium','Energy efficiency / retrofit -> building energy-efficiency retrofit.'),
  (['recycl','circular economy','solid waste','waste management'], 'c40_0037','medium','Recycling / waste management -> segregated collection of recyclables.'),
  (['compost','organic waste'], 'icare_0064','medium','Organic waste / compost -> organic waste management strategy.'),
  (['redd','deforestation','native forest','sustainable forest'], 'ipcc_0052','medium','REDD+ / avoided deforestation -> reduce deforestation and degradation.'),
  (['reforest','afforest','forest restoration'], 'ipcc_0053','medium','Afforestation / reforestation -> forest restoration.'),
  (['wetland','peatland'], 'ipcc_0060','medium','Wetland protection -> protect & restore wetlands.'),
  (['green space','urban park','green area','urban green'], 'c40_0042','medium','Urban green space -> expand urban & peri-urban green spaces.'),
  (['biogas','methane capture'], 'icare_0031','medium','Wastewater biogas -> capture & use biogas at treatment plants.'),
  (['green hydrogen','hydrogen'], 'icare_0078','low','Green hydrogen -> decarbonize industrial feedstocks (loose).'),
]
def map_action(text):
    t=' '+str(text).lower()+' '
    for kws,aid,conf,rat in KW:
        if any(k in t for k in kws): return aid,conf,rat
    return '', 'none', 'No specific mitigation intervention keyword; broad or out-of-scope project.'

TEXTCOLS=['project_name', 'objective', 'sector', 'subsector']
def row_text(r): return ' '.join(str(r.get(c,'')) for c in TEXTCOLS)
chile=chile.copy()
chile['_text']=chile.apply(row_text, axis=1)
m=chile['_text'].apply(map_action)
chile['action_id']=[x[0] for x in m]; chile['mapping_confidence']=[x[1] for x in m]; chile['rationale']=[x[2] for x in m]
chile['action_name']=chile['action_id'].map(lambda a: acat.get(a,('',''))[0] if a else '')
chile[['project_id','project_name','action_id','mapping_confidence']].head(50)

In [ ]:
# Validate, export the PROJECT DATASET + the crosswalk, coverage chart
for a in chile.loc[chile.action_id!='','action_id'].unique(): assert a in acat, a
os.makedirs('data', exist_ok=True)
chile.drop(columns=['_text']).to_csv('data/idb_chile_projects.csv', index=False)
chile[['project_id','project_name']+[c for c in TEXTCOLS if c!='project_name']+['action_id','action_name','mapping_confidence','rationale']].to_csv('data/idb_chile_to_actions.csv', index=False)
cov=chile['mapping_confidence'].value_counts().reindex(['high','medium','low','none']).fillna(0).astype(int)
mapped=int(cov[['high','medium','low']].sum())
print(f'Chile projects={len(chile)} | mapped={mapped} | none={int(cov["none"])}')
print('actions reached:', sorted(chile.loc[chile.action_id!="","action_id"].unique()))
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(6,2.6)); cov.plot.bar(ax=ax,color=['#1a7a3a','#6bbf59','#d9b54a','#bbbbbb'])
ax.set_title('IDB Chile projects by mapping confidence'); ax.set_ylabel('projects')
fig.tight_layout(); fig

In [ ]:
# Mapper self-tests (independent of the data pull)
assert map_action('Electromobility for Public Transport electric buses TRANSPORT Urban transport')[0]=='c40_0023'
assert map_action('Energy Efficiency and Renewable Energy in Housing solar URBAN Energy efficiency')[0] in ('c40_0016','icare_0012')
assert map_action('Sustainable Forest Management and Restoration native forest restoration ENVIRONMENT Forestry')[0] in ('ipcc_0052','ipcc_0053')
assert map_action('Public Finance Management Modernization fiscal systems REFORM Public management')[1]=='none'
print('self-tests passed')

## Findings

- IDB has the strongest Latin America coverage of the three, but no climate query parameter, so mitigation relevance is keyword-scored on name/objective/sector/subsector, then filtered. Non-climate operations (public finance, modernization) drop out before mapping.
- The objective text is decisive: it carries the intervention detail the broad `sector` label lacks (for example 'electric buses'), which is why it is folded into the mapping text.
- Some IDB operations deliver GCF money (IDB is a GCF accredited entity, e.g. FP189 e-mobility), so an IDB row and a GCF row can describe the same programme; relate at the (country, sector) level, do not double-count.
- Adaptation out of scope; intermediated access; link-don't-attribute; medium/low/none await adjudication.